# 01c Cleaning and airport network
Applies the cleaning rule from the README and builds the airport edge table for RQ2. Cancelled and diverted flights are **kept** (labelled in `status`): RQ2 uses all scheduled flights and we compare cancellation rates between clusters. Only flights with an impossible scheduled duration are removed, and 2400 is mapped to 0 in the scheduled times.

**Before running:** select the "Python (ctds)" kernel.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # project root, so `src` can be imported

import pandas as pd
from IPython.display import display

from src.config import CLEAN_PARQUET
from src.cleaning import make_clean_parquet
from src.data_loading import load_flights
from src.network import make_edges_parquet

pd.set_option("display.float_format", "{:,.4f}".format)

## 1. Cleaning
A flight is removed if `crs_elapsed_time` is missing or <= 0, if its implied speed is above 650 mph, or if its scheduled duration is less than half or more than twice the median of its route (origin, dest). A flight can match more than one rule, so the rule counts can add up to more than the total.

In [2]:
report = make_clean_parquet()
display(report)

clean = load_flights(CLEAN_PARQUET, columns=["status", "crs_dep_time", "crs_arr_time"])
print(f"Clean flights: {len(clean):,}")
display(clean.status.value_counts().to_frame())
print("Max scheduled dep/arr time:", clean.crs_dep_time.max(), clean.crs_arr_time.max())

[clean_flights] runtime: 49.53 s | peak memory (Python/pandas): 1317.3 MB | peak memory (pyarrow pool): 1035.6 MB


,flights_flagged,share
duration missing or <= 0,2,0.0000
speed > 650 mph,5,0.0000
duration < 0.5x or > 2x route median,38,0.0000
total removed (any rule),41,0.0000


Clean flights: 7,079,040


,count
status,
completed,6965246
cancelled,96309
diverted,17485


Max scheduled dep/arr time: 2359 2359


## 2. Airport edges (RQ2)
Directed edges origin → dest from the clean data, months 1–6, all scheduled flights (cancelled and diverted included). `distance = 1 / flights` is the edge length for betweenness, so busy routes count as short.

In [3]:
edges = make_edges_parquet()
airports = pd.unique(edges[["origin", "dest"]].astype(str).values.ravel())
print(f"Edges: {len(edges):,} | airports: {len(airports)} | flights: {edges.flights.sum():,}")
display(edges.sort_values("flights", ascending=False).head(10))

[build_edges] runtime: 0.54 s | peak memory (Python/pandas): 171.8 MB | peak memory (pyarrow pool): 1028.8 MB
Edges: 6,545 | airports: 343 | flights: 3,461,313


,origin,dest,flights,distance
3876,OGG,HNL,5829,0.0002
3229,HNL,OGG,5819,0.0002
3326,LAX,SFO,5482,0.0002
4192,SFO,LAX,5460,0.0002
410,LGA,ORD,5366,0.0002
3477,ORD,LGA,5365,0.0002
3117,PHX,DEN,4846,0.0002
4490,DEN,PHX,4810,0.0002
2484,BOS,DCA,4746,0.0002
3436,DCA,BOS,4746,0.0002
